# Smart Traffic Light Double-DQN Training (Kaggle)

This notebook trains a stronger traffic-light policy for `truongNgn/smart-traffic-light-system`. It keeps a small idempotent patch cell after cloning so the environment matches Sahal et al. (2023) more closely: each selected green phase is held for 10 seconds, phase switches still insert 2s yellow + 2s all-red, and the agent uses Double DQN targets with gradient clipping.

**Before running:** in Kaggle notebook settings, set **Internet = On**. A GPU helps PyTorch, but this SUMO-heavy workload can still run on CPU because `libsumo` is used.

The final cell creates `dqn_eval_best.pt`, selected by deterministic evaluation metrics rather than raw training reward. Download that file first for local SUMO testing.

## 1. Install SUMO and clone the repo

In [ ]:
!pip install -q eclipse-sumo traci sumolib libsumo

In [ ]:
!git clone --branch dev-truong --single-branch https://github.com/truongNgn/smart-traffic-light-system.git
%cd smart-traffic-light-system


## 1.1. Verify or patch training logic

dev-truong is the default branch for this project. This cell is intentionally idempotent: it does nothing when the branch already contains the improved training code, and patches only if Kaggle cloned an older commit.


In [ ]:
# Patch the cloned repo for a stronger, paper-aligned training run.
# Safe to rerun: every replacement is idempotent.
from pathlib import Path


def replace_once(path, old, new):
    path = Path(path)
    text = path.read_text(encoding="utf-8")
    if new in text:
        return
    if old not in text:
        raise RuntimeError(f"Patch target not found in {path}: {old[:80]!r}")
    path.write_text(text.replace(old, new, 1), encoding="utf-8")


constants = Path("common/constants.py")
text = constants.read_text(encoding="utf-8")
if "GREEN_DURATION_S" not in text:
    text = text.replace(
        "# --- Mandatory phase-switch safety buffers --------------------------------\nYELLOW_DURATION_S: float = 2.0",
        "# --- Mandatory phase-switch safety buffers --------------------------------\nGREEN_DURATION_S: float = 10.0\n\"\"\"Minimum green phase held for each agent action, matching Sahal et al. 2023.\"\"\"\n\nYELLOW_DURATION_S: float = 2.0",
    )
    constants.write_text(text, encoding="utf-8")

replace_once(
    "rl/env/traffic_env.py",
    "    DEFAULT_STEP_LENGTH_S,\n    GRID_CELLS_TOTAL,",
    "    DEFAULT_STEP_LENGTH_S,\n    GREEN_DURATION_S,\n    GRID_CELLS_TOTAL,",
)
replace_once(
    "rl/env/traffic_env.py",
    "        episode_duration_s: float = 3600.0,\n        initial_direction: Direction = Direction.EAST,",
    "        episode_duration_s: float = 3600.0,\n        green_duration_s: float = GREEN_DURATION_S,\n        initial_direction: Direction = Direction.EAST,",
)
replace_once(
    "rl/env/traffic_env.py",
    "        self.episode_duration_s = episode_duration_s\n        self.initial_direction = initial_direction",
    "        self.episode_duration_s = episode_duration_s\n        self.green_duration_s = green_duration_s\n        self.initial_direction = initial_direction",
)
replace_once(
    "rl/env/traffic_env.py",
    "        self._yellow_steps = max(1, round(YELLOW_DURATION_S / step_length_s))",
    "        self._green_steps = max(1, round(green_duration_s / step_length_s))\n        self._yellow_steps = max(1, round(YELLOW_DURATION_S / step_length_s))",
)
replace_once(
    "rl/env/traffic_env.py",
    "        else:\n            self._step_and_track()",
    "        else:\n            self._step_and_track(self._green_steps)",
)
replace_once(
    "rl/env/traffic_env.py",
    "        )\n        self._step_and_track()\n\n    def _step_and_track",
    "        )\n        self._step_and_track(self._green_steps)\n\n    def _step_and_track",
)
replace_once(
    "rl/train/config.py",
    "    episode_duration_s: float = Field(default=3600.0)\n    backend: str = Field(",
    "    episode_duration_s: float = Field(default=3600.0)\n    green_duration_s: float = Field(\n        default=10.0,\n        gt=0.0,\n        description=\"Seconds to hold each selected green action; 10s matches the paper.\",\n    )\n    backend: str = Field(",
)
replace_once(
    "rl/train/train.py",
    "        episode_duration_s=cfg.episode_duration_s,\n        backend=backend,",
    "        episode_duration_s=cfg.episode_duration_s,\n        green_duration_s=cfg.green_duration_s,\n        backend=backend,",
)
replace_once(
    "rl/agent/dqn_agent.py",
    "        with torch.no_grad():\n            next_q_values = self.target_net(next_states).max(dim=1).values\n            targets = rewards + self.gamma * next_q_values * (1.0 - dones)",
    "        with torch.no_grad():\n            next_actions = self.policy_net(next_states).argmax(dim=1, keepdim=True)\n            next_q_values = self.target_net(next_states).gather(1, next_actions).squeeze(1)\n            targets = rewards + self.gamma * next_q_values * (1.0 - dones)",
)
replace_once(
    "rl/agent/dqn_agent.py",
    "        self.optimizer.zero_grad()\n        loss.backward()\n        self.optimizer.step()",
    "        self.optimizer.zero_grad()\n        loss.backward()\n        nn.utils.clip_grad_norm_(self.policy_net.parameters(), max_norm=10.0)\n        self.optimizer.step()",
)

print("Training patch applied: 10s green hold + Double DQN target + gradient clipping.")


In [ ]:
# torch is already preinstalled on Kaggle's GPU image - don't reinstall it
!pip install -q pydantic pydantic-settings structlog gymnasium numpy tqdm

In [ ]:
import os
import sumo

# eclipse-sumo bundles its own binaries + tools/ under the installed
# package directory - point SUMO_HOME there instead of a system path.
os.environ["SUMO_HOME"] = os.path.dirname(sumo.__file__)
print("SUMO_HOME =", os.environ["SUMO_HOME"])

!sumo --version

If `sumo --version` fails to print a version here, the `eclipse-sumo` wheel most likely didn't ship a binary for this exact platform. Fall back to the apt-get route as a last resort (slower, and known to segfault on some Kaggle images - see the note in cell 1):

```bash
!apt-get update -qq && apt-get install -y -qq sumo sumo-tools sumo-doc
```
```python
import os
os.environ["SUMO_HOME"] = "/usr/share/sumo"
```

## 2. Confirm GPU is visible to PyTorch, and libsumo is importable

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

import libsumo
print("libsumo importable OK - training will use the fast in-process backend")

## 3. Build the SUMO network and generate demand

In [ ]:
!python -m simulation.net.build_net
!python -m simulation.net.generate_routes --duration 3600 --seed 42

## 4. Quick pipeline smoke test (optional but recommended)

Runs 2 tiny episodes end-to-end before committing to a long training run - catches setup problems in seconds instead of hours.

In [ ]:
from rl.train.config import TrainingConfig
from rl.train.train import train

smoke_cfg = TrainingConfig(
    num_episodes=2,
    episode_duration_s=60,
    checkpoint_dir="/kaggle/working/smoke_checkpoints",
    min_replay_size=8,
    batch_size=4,
)
_ = train(smoke_cfg)
print("Smoke test OK")

## 5. Train the improved policy

Recommended starting point: 1500 episodes, 3600s per episode, 10s minimum green, Double DQN target, and slower epsilon decay. Because each action now holds green for 10 seconds, each episode has far fewer control decisions than the previous 1-second version, so 1500 episodes is practical on Kaggle.

The raw `dqn_best.pt` is still saved during training, but do **not** blindly deploy it. The next section evaluates candidate checkpoints and writes `dqn_eval_best.pt`, which is the file you should bring back to the local repo first.

In [ ]:
from rl.train.config import TrainingConfig
from rl.train.train import train

cfg = TrainingConfig(
    num_episodes=1500,
    episode_duration_s=3600,
    green_duration_s=10.0,
    checkpoint_dir="/kaggle/working/checkpoints",
    batch_size=128,
    min_replay_size=5000,
    replay_capacity=100_000,
    learning_rate=5e-5,
    gamma=0.99,
    epsilon_start=1.0,
    epsilon_end=0.02,
    epsilon_decay_episodes=1200,
    target_sync_every_episodes=5,
    checkpoint_every_episodes=50,
    train_every_n_steps=4,
    log_every_episodes=10,
)
agent = train(cfg)


## 6. Resuming (only needed if a session got cut off)

Kaggle sessions get killed after ~9-12 hours. `/kaggle/working/` output persists between sessions, so if training didn't finish, start a new session and continue from the last checkpoint instead of restarting from scratch.

In [ ]:
# from rl.train.config import TrainingConfig
# from rl.train.train import train
#
# resumed_cfg = TrainingConfig(
#     num_episodes=1000,
#     episode_duration_s=3600,
#     checkpoint_dir="/kaggle/working/checkpoints",
#     resume_from="/kaggle/working/checkpoints/dqn_final.pt",
# )
# agent = train(resumed_cfg)

## 7. Evaluate checkpoints and select the deployment model

This evaluates recent periodic checkpoints plus `dqn_best.pt` and `dqn_final.pt` on held-out seeds. The selected checkpoint is copied to `/kaggle/working/checkpoints/dqn_eval_best.pt`. Lower score is better; the score favors lower mean/final waiting time, lower queue length, and higher throughput.

In [ ]:
from pathlib import Path
import shutil

from benchmark.policies import DQNPolicy, FixedTimePolicy
from benchmark.run_episode import run_episode
from rl.agent.dqn_agent import DQNAgent
from rl.env.traffic_env import SumoTrafficEnv
from rl.train.checkpoint import load_checkpoint

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
EVAL_DURATION_S = 1200
EVAL_SEEDS = [101, 102, 103]

periodic = sorted(CHECKPOINT_DIR.glob("dqn_episode_*.pt"), key=lambda p: int(p.stem.split("_")[-1]))
candidate_paths = []
for path in [CHECKPOINT_DIR / "dqn_best.pt", CHECKPOINT_DIR / "dqn_final.pt", *periodic[-10:]]:
    if path.exists() and path not in candidate_paths:
        candidate_paths.append(path)


def mean(values):
    return sum(values) / len(values) if values else 0.0


def aggregate(metrics):
    dicts = [m.to_dict() for m in metrics]
    return {key: mean([d[key] for d in dicts]) for key in dicts[0]}


def score(metrics):
    # Lower is better. Throughput is subtracted because more arrivals are good.
    return (
        metrics["mean_waiting_time_s"]
        + 0.5 * metrics["final_waiting_time_s"]
        + 25.0 * metrics["mean_queue_length"]
        - 2.0 * metrics["arrived_vehicles"]
    )


env = SumoTrafficEnv(
    sumocfg_path="simulation/net/intersection.sumocfg",
    episode_duration_s=EVAL_DURATION_S,
    green_duration_s=10.0,
    backend="libsumo",
)
try:
    fixed_metrics = [run_episode(env, FixedTimePolicy(green_duration_s=20.0), seed=s) for s in EVAL_SEEDS]
finally:
    env.close()
fixed_agg = aggregate(fixed_metrics)
print("Fixed-time baseline:", fixed_agg)

results = []
for path in candidate_paths:
    agent = DQNAgent()
    trained_episode = load_checkpoint(path, agent)
    policy = DQNPolicy(agent, epsilon=0.0)
    env = SumoTrafficEnv(
        sumocfg_path="simulation/net/intersection.sumocfg",
        episode_duration_s=EVAL_DURATION_S,
        green_duration_s=10.0,
        backend="libsumo",
    )
    try:
        metrics = [run_episode(env, policy, seed=s) for s in EVAL_SEEDS]
    finally:
        env.close()
    agg = aggregate(metrics)
    agg_score = score(agg)
    results.append((agg_score, path, trained_episode, agg))
    print(
        f"{path.name:<22} ep={trained_episode:<5} score={agg_score:9.2f} "
        f"mean_wait={agg['mean_waiting_time_s']:8.2f} queue={agg['mean_queue_length']:6.2f} "
        f"arrived={agg['arrived_vehicles']:7.2f}"
    )

results.sort(key=lambda row: row[0])
best_score, best_path, best_episode, best_metrics = results[0]
selected_path = CHECKPOINT_DIR / "dqn_eval_best.pt"
shutil.copy2(best_path, selected_path)
print("\nSelected:", best_path.name, "episode", best_episode, "score", best_score)
print("Metrics:", best_metrics)
print("Wrote:", selected_path)


## 8. Download the selected model

Download `dqn_eval_best.pt` first. It is selected by deterministic evaluation across held-out seeds, so it is usually a better deployment candidate than the raw training `dqn_best.pt`. Keep `dqn_best.pt` and `dqn_final.pt` too if you want to compare locally.

In [ ]:
!ls -lh /kaggle/working/checkpoints


In [ ]:
import base64
from pathlib import Path
from IPython.display import HTML, display


def download_link(path, filename=None):
    filename = filename or path.split("/")[-1]
    with open(path, "rb") as f:
        data = f.read()
    b64 = base64.b64encode(data).decode()
    return HTML(f'<a download="{filename}" href="data:application/octet-stream;base64,{b64}">Download {filename}</a>')


for path in [
    "/kaggle/working/checkpoints/dqn_eval_best.pt",
    "/kaggle/working/checkpoints/dqn_best.pt",
    "/kaggle/working/checkpoints/dqn_final.pt",
]:
    if Path(path).exists():
        display(download_link(path))
